# XGBoost, Random Forest and LightGBM - Fixed Evaluation

This notebook replaces the previous tuning flow where CV failures were hidden instead of being raised.

Corrections:
- temporal train/validation/test split;
- `RandomizedSearchCV(error_score="raise")`, so failed CV folds cannot silently become `nan`;
- SMOTE is inside an `imblearn.Pipeline` and therefore fit only on training folds;
- validation set chooses the classification threshold;
- the final model is refit on train+validation before test evaluation;
- final metrics are reported on the untouched test set;
- `duration` is evaluated separately as benchmark vs realistic scenario.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", message="X does not have valid feature names")

RANDOM_STATE = 42
N_ITER = 8
CV_SPLITS = 3


In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Models" / "processed_bank_data.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate Models/processed_bank_data.csv")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "Models" / "processed_bank_data.csv"

df = pd.read_csv(DATA_PATH)
print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print("Positive rate:", f"{df['y'].mean():.2%}")
display(df.head())


Data path: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\Models\processed_bank_data.csv
Shape: (41176, 58)
Positive rate: 11.27%


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,age_group_adult,age_group_middle,age_group_senior,y
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0


In [3]:
def temporal_split(frame: pd.DataFrame, target: str = "y", train_size: float = 0.7, valid_size: float = 0.1):
    n = len(frame)
    train_end = int(n * train_size)
    valid_end = int(n * (train_size + valid_size))
    train = frame.iloc[:train_end].copy()
    valid = frame.iloc[train_end:valid_end].copy()
    test = frame.iloc[valid_end:].copy()

    splits = {}
    for name, part in [("train", train), ("valid", valid), ("test", test)]:
        X = part.drop(columns=[target]).astype(float)
        y = part[target].astype(int)
        splits[name] = (X, y)
        print(f"{name:5s}: {len(part):5d} rows | positive rate = {y.mean():.2%} | positives = {int(y.sum())}")
    return splits

splits = temporal_split(df)
X_train, y_train = splits["train"]
X_valid, y_valid = splits["valid"]
X_test, y_test = splits["test"]
X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)
print(f"train+valid: {len(X_train_valid):5d} rows | positive rate = {y_train_valid.mean():.2%} | positives = {int(y_train_valid.sum())}")

scenarios = {
    "Scenario 1 - With duration (benchmark)": list(X_train.columns),
    "Scenario 2 - Without duration (realistic)": [col for col in X_train.columns if col != "duration"],
}


train: 28823 rows | positive rate = 5.58% | positives = 1607
valid:  4117 rows | positive rate = 11.97% | positives = 493
test :  8236 rows | positive rate = 30.83% | positives = 2539
train+valid: 32940 rows | positive rate = 6.38% | positives = 2100


In [4]:
def choose_threshold(y_true, y_proba, thresholds=np.arange(0.05, 0.96, 0.01)):
    rows = []
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(round(threshold, 2)),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            }
        )
    table = pd.DataFrame(rows)
    best = table.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    return float(best["threshold"]), table


def score_row(model_name, scenario_name, split_name, y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "model": model_name,
        "scenario": scenario_name,
        "split": split_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def make_pipeline(estimator):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            ("model", estimator),
        ]
    )


In [5]:
model_spaces = {
    "XGBoost": {
        "estimator": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "params": {
            "model__n_estimators": [120, 200, 300],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.03, 0.07, 0.12],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
            "model__min_child_weight": [1, 5],
            "model__reg_lambda": [1.0, 2.0],
        },
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        "params": {
            "model__n_estimators": [150, 250, 350],
            "model__max_depth": [8, 12, None],
            "model__min_samples_split": [2, 8, 16],
            "model__min_samples_leaf": [2, 5, 10],
            "model__max_features": ["sqrt", 0.5],
            "model__class_weight": [None, "balanced"],
        },
    },
    "LightGBM": {
        "estimator": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=1, verbose=-1),
        "params": {
            "model__n_estimators": [120, 200, 300],
            "model__max_depth": [-1, 6, 10],
            "model__learning_rate": [0.03, 0.07, 0.12],
            "model__num_leaves": [31, 63, 127],
            "model__min_child_samples": [20, 60, 100],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
            "model__reg_lambda": [0.0, 1.0, 2.0],
        },
    },
}


In [6]:
def tune_model(model_name, estimator, param_distributions, X, y):
    cv = TimeSeriesSplit(n_splits=CV_SPLITS)
    search = RandomizedSearchCV(
        estimator=make_pipeline(estimator),
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score="raise",
        verbose=1,
        refit=True,
    )
    search.fit(X, y)
    print(f"{model_name} best CV ROC-AUC: {search.best_score_:.4f}")
    print("Best params:", search.best_params_)
    return search

results = []
fitted = {}
threshold_tables = {}

for scenario_name, feature_cols in scenarios.items():
    print("\n" + "#" * 100)
    print(scenario_name)
    X_tr = X_train[feature_cols]
    X_val = X_valid[feature_cols]
    X_te = X_test[feature_cols]

    for model_name, spec in model_spaces.items():
        print("\n" + "=" * 80)
        print(model_name)
        search = tune_model(model_name, spec["estimator"], spec["params"], X_tr, y_train)

        valid_proba = search.predict_proba(X_val)[:, 1]
        best_threshold, threshold_table = choose_threshold(y_valid, valid_proba)

        # Refit the selected pipeline on all past data before the test period.
        final_model = search.best_estimator_
        final_model.fit(X_train_valid[feature_cols], y_train_valid)
        test_proba = final_model.predict_proba(X_te)[:, 1]

        results.append(score_row(model_name, scenario_name, "validation_at_0.50", y_valid, valid_proba, 0.50))
        results.append(score_row(model_name, scenario_name, "validation_tuned", y_valid, valid_proba, best_threshold))
        results.append(score_row(model_name, scenario_name, "test_at_0.50_after_refit", y_test, test_proba, 0.50))
        results.append(score_row(model_name, scenario_name, "test_with_validation_threshold_after_refit", y_test, test_proba, best_threshold))

        key = (model_name, scenario_name)
        fitted[key] = {"model": final_model, "features": feature_cols, "threshold": best_threshold, "search": search}
        threshold_tables[key] = threshold_table

summary_df = pd.DataFrame(results)
display(summary_df.round(4))



####################################################################################################
Scenario 1 - With duration (benchmark)

XGBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
XGBoost best CV ROC-AUC: 0.8974
Best params: {'model__subsample': 1.0, 'model__reg_lambda': 2.0, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 7, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}

Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Random Forest best CV ROC-AUC: 0.9125
Best params: {'model__n_estimators': 250, 'model__min_samples_split': 16, 'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': 8, 'model__class_weight': 'balanced'}

LightGBM
Fitting 3 folds for each of 8 candidates, totalling 24 fits
LightGBM best CV ROC-AUC: 0.8995
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 0.0, 'model__num_leaves': 31, 'model__n_estimators': 200, 'model__min_child_samples'

,model,scenario,split,threshold,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy
0,XGBoost,Scenario 1 - With duration (benchmark),validation_at_0.50,0.50,0.8235,0.3861,0.4179,0.3306,0.3692,0.6340
1,XGBoost,Scenario 1 - With duration (benchmark),validation_tuned,0.31,0.8235,0.3861,0.3514,0.6065,0.4449,0.7271
2,XGBoost,Scenario 1 - With duration (benchmark),test_at_0.50_after_refit,0.50,0.7861,0.5492,0.6111,0.2296,0.3338,0.5822
3,XGBoost,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.31,0.7861,0.5492,0.6129,0.3367,0.4347,0.6210
4,Random Forest,Scenario 1 - With duration (benchmark),validation_at_0.50,0.50,0.8078,0.3533,0.3057,0.6207,0.4096,0.7145
5,Random Forest,Scenario 1 - With duration (benchmark),validation_tuned,0.52,0.8078,0.3533,0.3310,0.5801,0.4215,0.7103
6,Random Forest,Scenario 1 - With duration (benchmark),test_at_0.50_after_refit,0.50,0.8331,0.6114,0.6247,0.4904,0.5494,0.6795
7,Random Forest,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.52,0.8331,0.6114,0.6236,0.4482,0.5215,0.6638
8,LightGBM,Scenario 1 - With duration (benchmark),validation_at_0.50,0.50,0.8398,0.4367,0.4535,0.4746,0.4638,0.6984
9,LightGBM,Scenario 1 - With duration (benchmark),validation_tuned,0.42,0.8398,0.4367,0.3978,0.5923,0.4760,0.7352


In [7]:
final_test = summary_df[summary_df["split"].eq("test_with_validation_threshold_after_refit")]
final_test = final_test.sort_values(["scenario", "pr_auc", "f1"], ascending=[True, False, False])
display(final_test.round(4))

for _, row in final_test.iterrows():
    key = (row["model"], row["scenario"])
    info = fitted[key]
    proba = info["model"].predict_proba(X_test[info["features"]])[:, 1]
    pred = (proba >= info["threshold"]).astype(int)
    print("=" * 100)
    print(f"{row['model']} | {row['scenario']} | threshold={info['threshold']:.2f}")
    print(classification_report(y_test, pred, target_names=["No", "Yes"], zero_division=0))


,model,scenario,split,threshold,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy
7,Random Forest,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.52,0.8331,0.6114,0.6236,0.4482,0.5215,0.6638
3,XGBoost,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.31,0.7861,0.5492,0.6129,0.3367,0.4347,0.6210
11,LightGBM,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.42,0.7853,0.5315,0.6006,0.3127,0.4113,0.6100
19,Random Forest,Scenario 2 - Without duration (realistic),test_with_validation_threshold_after_refit,0.11,0.7321,0.5134,0.3742,0.9275,0.5332,0.6181
15,XGBoost,Scenario 2 - Without duration (realistic),test_with_validation_threshold_after_refit,0.22,0.6979,0.4926,0.5378,0.3923,0.4537,0.6210
23,LightGBM,Scenario 2 - Without duration (realistic),test_with_validation_threshold_after_refit,0.23,0.6289,0.4196,0.4597,0.3167,0.3750,0.5754


Random Forest | Scenario 1 - With duration (benchmark) | threshold=0.52
              precision    recall  f1-score   support

          No       0.78      0.88      0.83      5697
         Yes       0.62      0.45      0.52      2539

    accuracy                           0.75      8236
   macro avg       0.70      0.66      0.67      8236
weighted avg       0.73      0.75      0.73      8236

XGBoost | Scenario 1 - With duration (benchmark) | threshold=0.31
              precision    recall  f1-score   support

          No       0.75      0.91      0.82      5697
         Yes       0.61      0.34      0.43      2539

    accuracy                           0.73      8236
   macro avg       0.68      0.62      0.63      8236
weighted avg       0.71      0.73      0.70      8236

LightGBM | Scenario 1 - With duration (benchmark) | threshold=0.42
              precision    recall  f1-score   support

          No       0.75      0.91      0.82      5697
         Yes       0.60      0.3

In [8]:
# Feature importance for the best realistic model by PR-AUC.
realistic = final_test[final_test["scenario"].str.contains("Without duration")].copy()
best_realistic = realistic.sort_values(["pr_auc", "f1"], ascending=False).iloc[0]
key = (best_realistic["model"], best_realistic["scenario"])
info = fitted[key]
best_estimator = info["model"].named_steps["model"]
feature_cols = info["features"]

if hasattr(best_estimator, "feature_importances_"):
    importance_df = pd.DataFrame(
        {"feature": feature_cols, "importance": best_estimator.feature_importances_}
    ).sort_values("importance", ascending=False)
    print("Best realistic model:", key)
    display(importance_df.head(20))
else:
    print("Selected model does not expose feature_importances_.")


Best realistic model: ('Random Forest', 'Scenario 2 - Without duration (realistic)')


,feature,importance
1,campaign,0.262740
34,housing_yes,0.095711
7,euribor3m,0.092977
31,default_unknown,0.067346
6,cons.conf.idx,0.057531
0,age,0.054275
37,contact_telephone,0.026661
50,day_of_week_wed,0.026493
48,day_of_week_thu,0.021818
36,loan_yes,0.021740


## Reporting Rule

For the final presentation, use the rows where `split == "test_with_validation_threshold_after_refit"`. These rows use a threshold selected on validation data and then evaluated once on the untouched temporal test set.
